# Milestone 2: Baseline + Main Model

Trains two credit risk models on the Home Credit dataset and compares them on a shared evaluation framework.

- **Baseline**: Logistic Regression on WoE-encoded features — the production-standard scorecard approach, interpretable as a points-based score and regulator-friendly.
- **Challenger**: LightGBM on raw features — modern gradient-boosting that handles missingness natively and learns non-linear interactions without manual engineering.

Both models use the same train/val/test splits and evaluation metrics (AUC, KS, Brier). The comparison establishes whether modern ML adds meaningful value over the traditional approach for this dataset.

Final feature set: 40 features (39 without `EXT_SOURCE_1`). Train/val/test = 70/15/15, stratified on TARGET to preserve the 8% default rate.

## 1. Setup

Load the joined application + bureau data, apply basic cleaning, retrieve the cached train/val/test split, and prepare the modeling feature set. The split indices are produced once in `src/splits.py` and reused across all milestones.

In [ ]:
import sys
import pandas as pd
import numpy as np

from src.data import load_joined, basic_clean
from src.splits import get_splits, split_report
from src.features import prepare_features, get_feature_columns

df = load_joined()
df = basic_clean(df)

train_idx, val_idx, test_idx = get_splits(df)
print(split_report(df))

In [ ]:
# Prepare features (with EXT_SOURCE_1 included for the main model)
df_features = prepare_features(df, include_ext_source_1=True)

# Split into train/val/test using the indices we saved
train = df_features.loc[train_idx]
val = df_features.loc[val_idx]
test = df_features.loc[test_idx]

# Define feature list and identify categorical columns
feature_cols = get_feature_columns(include_ext_source_1=True)

categorical_cols = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
    "NAME_HOUSING_TYPE",
    "OCCUPATION_TYPE",
]

# Separate features and target for each split
X_train, y_train = train[feature_cols], train["TARGET"]
X_val, y_val = val[feature_cols], val["TARGET"]
X_test, y_test = test[feature_cols], test["TARGET"]

print(f"Train: {X_train.shape}, default rate {y_train.mean():.4f}")
print(f"Val:   {X_val.shape}, default rate {y_val.mean():.4f}")
print(f"Test:  {X_test.shape}, default rate {y_test.mean():.4f}")

## 2. WoE Encoding

**Weight of Evidence (WoE)** is the production-standard feature encoding for credit scorecards. Each feature is binned into ranges (continuous) or category groups (categorical), and each bin is assigned a WoE value computed from the bad-rate-to-good-rate ratio within that bin.

WoE encoding has four properties that make it the right tool for the baseline:

1. **Handles non-linearity automatically.** The U-shaped pattern in `bureau_count` (low default rate in the middle, elevated at both extremes) gets captured directly by the bin-level WoE values, even though logistic regression itself is linear.
2. **Handles missing values cleanly.** NaN is treated as its own bin and gets its own WoE — preserving the missingness-as-signal pattern surfaced in EDA Section 4.
3. **Robust to outliers.** Extreme values (e.g., `loan_to_income` reaching 84x) get bucketed into the top bin; their leverage on the model is bounded.
4. **Produces a readable scorecard.** Each bin's WoE × the model coefficient gives a points contribution. The final model can be expressed as a points-based scorecard, which is how most production credit models are documented for compliance.

Implementation uses `optbinning`, which selects bin edges by maximizing Information Value (IV) subject to constraints on bin size and monotonicity.

In [ ]:
from optbinning import BinningProcess

# Set up the binning process
# - variable_names: list of all features
# - categorical_variables: which ones are categorical (rest are numerical)
# - min_n_bins / max_n_bins: control granularity
binning_process = BinningProcess(
    variable_names=feature_cols,
    categorical_variables=categorical_cols,
    min_n_bins=3,
    max_n_bins=8,
    min_bin_size=0.05,  # each bin must have at least 5% of training data
)

# IMPORTANT: fit ONLY on training data. This prevents leakage.
binning_process.fit(X_train, y_train)

In [ ]:
# Look at the binning table for a specific feature
# Information Value (IV) tells us how predictive each feature is
summary = binning_process.summary()
print(summary[["name", "iv", "n_bins"]].sort_values("iv", ascending=False).head(15))

### Inspect binning tables

Three features worth looking at in detail to understand what WoE binning produces:

- **`EXT_SOURCE_3`**: monotonic feature, the strongest predictor. Default rate drops smoothly as score increases.
- **`bureau_count`**: U-shaped feature from EDA — both extremes (0 records and 17+) carry elevated risk. WoE binning should preserve this non-linearity.
- **`payment_to_income`**: feature where binning surfaces patterns not visible in raw decile analysis (see note below).

In [ ]:
# Get the binning table for the strongest feature (EXT_SOURCE_3)
optb = binning_process.get_binned_variable("EXT_SOURCE_3")
print(optb.binning_table.build())

In [ ]:
optb = binning_process.get_binned_variable("bureau_count")
print(optb.binning_table.build())

In [ ]:
optb = binning_process.get_binned_variable("payment_to_income")
print(optb.binning_table.build())

**Note on PTI:** Binning preserves the non-monotonic pattern at the top decile observed in EDA
(high PTI applicants are slightly safer than upper-mid PTI, likely a selection effect from
intake screening). Model handles this correctly via WoE; no action taken at the baseline.
Revisit if LightGBM importance or fair-lending analysis flags it.

In [ ]:
# Transform applies the LEARNED bins from training to all splits
X_train_woe = binning_process.transform(X_train)
X_val_woe = binning_process.transform(X_val)
X_test_woe = binning_process.transform(X_test)

print(f"WoE-encoded train shape: {X_train_woe.shape}")
print(f"\nOriginal values replaced with bin WoE values (mostly small floats around 0):")
print(X_train_woe[["EXT_SOURCE_3", "age_years", "bureau_count"]].describe().round(3))

## 3. Baseline: Logistic Regression on WoE Features

Standard L2-regularized logistic regression on the WoE-encoded feature set. All features are now on a comparable scale (mostly small floats centered around 0), so no additional scaling is needed.

Evaluation metrics chosen for credit risk context:
- **AUC**: standard ML ranking metric
- **KS statistic**: max separation between defaulter and non-defaulter score distributions; the credit risk industry's preferred metric
- **Brier score**: mean squared error of probabilities; measures calibration, which matters for pricing and expected-loss calculations

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from scipy.stats import ks_2samp


def ks_statistic(y_true, y_pred):
    """KS statistic: max separation between defaulter and non-defaulter score distributions."""
    return ks_2samp(y_pred[y_true == 1], y_pred[y_true == 0]).statistic


# Train baseline: L2-regularized LR on WoE-encoded features
lr_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr_model.fit(X_train_woe, y_train)

# Predict on validation set
y_val_pred_lr = lr_model.predict_proba(X_val_woe)[:, 1]

# Evaluate
auc_val = roc_auc_score(y_val, y_val_pred_lr)
ks_val = ks_statistic(y_val.values, y_val_pred_lr)
brier_val = brier_score_loss(y_val, y_val_pred_lr)

print(f"Baseline (WoE + LR) validation metrics:")
print(f"  AUC:   {auc_val:.4f}")
print(f"  KS:    {ks_val:.4f}")
print(f"  Brier: {brier_val:.4f}")

In [ ]:
# Top features by absolute coefficient magnitude
# Note: all coefficients should be negative under optbinning's WoE convention.
# Higher WoE = safer applicant, so the LR coefficient runs in the opposite direction.
coefs = pd.DataFrame({
    "feature": feature_cols,
    "coef": lr_model.coef_[0],
}).sort_values("coef", key=abs, ascending=False)

print("Top 15 baseline features by coefficient:")
print(coefs.head(15))

## 4. Challenger: LightGBM on Raw Features

LightGBM is a gradient-boosting tree model that builds many shallow decision trees sequentially, each correcting the residual errors of the previous ensemble.

Key differences from the baseline:
- **No WoE encoding** — trees use raw feature values directly.
- **Handles NaN natively** — tree splits learn to route missing values to the appropriate branch.
- **Captures interactions automatically** — deep tree paths represent combinations of features without manual engineering.
- **No class weighting** — the natural 8% imbalance is fine for gradient boosting on this much data, and `scale_pos_weight` hurts calibration more than it helps AUC.

Hyperparameters are sensible defaults rather than tuned for max AUC. The project optimizes for framing and credit-risk methodology, not last-mile performance. Early stopping on the validation set prevents overfitting.

In [ ]:
import lightgbm as lgb

# LightGBM needs categorical columns to be the pandas 'category' dtype
# (or you can pass column names via the categorical_feature parameter)
X_train_lgb = X_train.copy()
X_val_lgb = X_val.copy()
X_test_lgb = X_test.copy()

for col in categorical_cols:
    X_train_lgb[col] = X_train_lgb[col].astype("category")
    X_val_lgb[col] = X_val_lgb[col].astype("category")
    X_test_lgb[col] = X_test_lgb[col].astype("category")

print("Categorical columns converted to 'category' dtype.")

In [ ]:
# Train the model
# Sensible defaults; not tuned. No class weighting — let LightGBM see the natural
# imbalance so it outputs well-calibrated probabilities.

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "random_state": 42,
    "verbose": -1,
}

lgb_train = lgb.Dataset(X_train_lgb, label=y_train, categorical_feature=categorical_cols)
lgb_val = lgb.Dataset(X_val_lgb, label=y_val, categorical_feature=categorical_cols, reference=lgb_train)

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=2000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(period=100),
    ],
)

print(f"\nBest iteration: {lgb_model.best_iteration}")

In [ ]:
# Evaluate on the validation set
y_val_pred_lgb = lgb_model.predict(X_val_lgb, num_iteration=lgb_model.best_iteration)

auc_val_lgb = roc_auc_score(y_val, y_val_pred_lgb)
ks_val_lgb = ks_statistic(y_val.values, y_val_pred_lgb)
brier_val_lgb = brier_score_loss(y_val, y_val_pred_lgb)

print(f"LightGBM validation metrics:")
print(f"  AUC:   {auc_val_lgb:.4f}")
print(f"  KS:    {ks_val_lgb:.4f}")
print(f"  Brier: {brier_val_lgb:.4f}")

In [ ]:
# Feature importance — using 'gain' which measures how much each feature reduced loss
importance = pd.DataFrame({
    "feature": X_train_lgb.columns,
    "importance": lgb_model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)

print("Top 15 features by LightGBM gain:")
print(importance.head(15))

## 5. Comparison and Calibration Check

Side-by-side comparison of the two models on validation, plus a calibration sanity check. A well-calibrated model's mean predicted PD should match the actual default rate of the evaluation set — this is a necessary (not sufficient) condition for using the probabilities in pricing or expected-loss calculations downstream.

In [ ]:
# Side-by-side metrics
print("=== Side-by-side: validation metrics ===")
print(f"{'Metric':<8} {'Baseline (LR)':>16} {'LightGBM':>12} {'Delta':>10}")
print(f"{'AUC':<8} {auc_val:>16.4f} {auc_val_lgb:>12.4f} {auc_val_lgb - auc_val:>+10.4f}")
print(f"{'KS':<8} {ks_val:>16.4f} {ks_val_lgb:>12.4f} {ks_val_lgb - ks_val:>+10.4f}")
print(f"{'Brier':<8} {brier_val:>16.4f} {brier_val_lgb:>12.4f} {brier_val_lgb - brier_val:>+10.4f}  (negative = better)")

# Calibration sanity check — both models' mean PD should match the true rate
print("\n=== Calibration sanity check ===")
print(f"  Mean predicted PD (LightGBM): {y_val_pred_lgb.mean():.4f}")
print(f"  Mean predicted PD (LR):       {y_val_pred_lr.mean():.4f}")
print(f"  Actual default rate (val):    {y_val.mean():.4f}")

### Milestone 2 Takeaways

| Metric | Baseline (WoE+LR) | LightGBM | Δ |
|---|---|---|---|
| AUC | 0.7462 | 0.7611 | +0.0149 |
| KS | 0.3713 | 0.3907 | +0.0193 |
| Brier | 0.0687 | 0.0677 | −0.0010 |

LightGBM trained for 134 boosting rounds before early stopping. Both models output well-calibrated probabilities (mean predicted PD matches the validation default rate to 3 decimal places).

**Feature importance overlap.** Top features by importance largely overlap between the two models. `EXT_SOURCE_2` and `EXT_SOURCE_3` dominate both (top-2 IV of 0.64 in the baseline; combined gain importance ~80,000 in LightGBM). `EXT_SOURCE_1`, employment tenure, financing premium, age, and occupation form the supporting cast in both models.

**Notable differences.**
- LightGBM surfaces `payment_to_income` and `DAYS_ID_PUBLISH` via tree interactions, where the linear baseline weights them less.
- The baseline's top coefficient is `payment_to_income` — the linear model leans hard on this engineered ratio for the signal LightGBM captures through interactions.

**Note on class imbalance.** Initial training with `scale_pos_weight=11.39` improved AUC marginally (+0.0008) but destroyed Brier (0.18 vs 0.07). For credit risk, calibrated probabilities matter more than marginal AUC, so the final model uses no class weighting.

**Decision for downstream work.** LightGBM beats the baseline by ~1.5 AUC points and ~2 KS points — meaningful but not transformative. Both models carry forward into Milestones 3-4: the baseline as a regulator-friendly fallback, LightGBM as the main model for calibration, segment analysis, threshold selection, and fair lending audit.